In [1]:
import os
import glob
import pandas as pd
from sqlalchemy import create_engine, text

# Set path relative to the 'notebooks' folder
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
DB_PATH = "../bluestock_mf.db"

os.makedirs(PROCESSED_DIR, exist_ok=True)
print("🧼 Starting Data Cleaning & Transformation...")

# 1. Clean nav_history.csv
nav_raw_path = os.path.join(RAW_DIR, "nav_history.csv")
if os.path.exists(nav_raw_path):
    df_nav = pd.read_csv(nav_raw_path)
    df_nav['date'] = pd.to_datetime(df_nav['date'])
    df_nav = df_nav.sort_values(by=['amfi_code', 'date'])
    
    # Forward-fill missing NAV values for holidays/weekends
    df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()
    df_nav = df_nav.drop_duplicates()
    df_nav = df_nav[df_nav['nav'] > 0]
    
    df_nav.to_csv(os.path.join(PROCESSED_DIR, "nav_history_clean.csv"), index=False)
    print("✅ nav_history.csv cleaned.")

# 2. Clean investor_transactions.csv
trans_raw_path = os.path.join(RAW_DIR, "investor_transactions.csv")
if os.path.exists(trans_raw_path):
    df_trans = pd.read_csv(trans_raw_path)
    df_trans['transaction_type'] = df_trans['transaction_type'].replace({
        'sip': 'SIP', 'SIP_Investment': 'SIP',
        'lumpsum': 'Lumpsum', 'one_time': 'Lumpsum',
        'redemption': 'Redemption', 'withdraw': 'Redemption'
    })
    df_trans = df_trans[df_trans['amount'] > 0]
    df_trans = df_trans[df_trans['KYC_status'].isin(['Verified', 'Pending', 'Failed'])]
    
    df_trans.to_csv(os.path.join(PROCESSED_DIR, "investor_transactions_clean.csv"), index=False)
    print("✅ investor_transactions.csv cleaned.")

# 3. Clean scheme_performance.csv
perf_raw_path = os.path.join(RAW_DIR, "scheme_performance.csv")
if os.path.exists(perf_raw_path):
    df_perf = pd.read_csv(perf_raw_path)
    df_perf['return_percentage'] = pd.to_numeric(df_perf['return_percentage'], errors='coerce')
    df_perf['expense_ratio_anomaly'] = ~(df_perf['expense_ratio'].between(0.1, 2.5))
    
    df_perf.to_csv(os.path.join(PROCESSED_DIR, "scheme_performance_clean.csv"), index=False)
    print("✅ scheme_performance.csv cleaned.")

🧼 Starting Data Cleaning & Transformation...
✅ nav_history.csv cleaned.


In [2]:
print("🗄️ Setting up SQLite Database and Schema...")
engine = create_engine(f"sqlite:///{DB_PATH}")

schema_sql = """
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    category TEXT,
    sub_category TEXT,
    risk_grade TEXT
);

CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY(amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id TEXT PRIMARY KEY,
    amfi_code INTEGER,
    user_id TEXT,
    transaction_type TEXT,
    amount REAL,
    date TEXT,
    KYC_status TEXT,
    FOREIGN KEY(amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    return_percentage REAL,
    expense_ratio REAL,
    FOREIGN KEY(amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with engine.connect() as conn:
    for statement in schema_sql.split(";"):
        if statement.strip():
            conn.execute(text(statement))
print("✅ SQLite Tables created successfully.")

🗄️ Setting up SQLite Database and Schema...
✅ SQLite Tables created successfully.


In [3]:
print("🚀 Loading data into SQLite...")

if os.path.exists(os.path.join(PROCESSED_DIR, "nav_history_clean.csv")):
    df_nav_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "nav_history_clean.csv"))
    df_nav_clean.to_sql("fact_nav", con=engine, if_exists="append", index=False)
    print(f"Loaded {len(df_nav_clean)} rows into fact_nav.")
    
if os.path.exists(os.path.join(PROCESSED_DIR, "investor_transactions_clean.csv")):
    df_trans_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "investor_transactions_clean.csv"))
    df_trans_clean.to_sql("fact_transactions", con=engine, if_exists="append", index=False)
    print(f"Loaded {len(df_trans_clean)} rows into fact_transactions.")

🚀 Loading data into SQLite...
Loaded 46000 rows into fact_nav.


In [4]:
print("📊 Running Analytical Insights...\n")

# Query 1: Top 5 funds by total transacted investment amount (AUM proxy)
q1 = """
SELECT amfi_code, SUM(amount) as total_aum 
FROM fact_transactions 
GROUP BY amfi_code 
ORDER BY total_aum DESC 
LIMIT 5;
"""
print("--- TOP 5 FUNDS BY AUM ---")
print(pd.read_sql_query(q1, con=engine), "\n")

# Query 2: Average NAV per Month
q2 = """
SELECT strftime('%Y-%m', date) as month, AVG(nav) as avg_nav 
FROM fact_nav 
GROUP BY month 
LIMIT 5;
"""
print("--- AVERAGE NAV PER MONTH (First 5 Rows) ---")
print(pd.read_sql_query(q2, con=engine), "\n")

# Query 3: Funds with an expense ratio less than 1%
q3 = """
SELECT amfi_code, expense_ratio 
FROM fact_performance 
WHERE expense_ratio < 1.0;
"""
print("--- LOW EXPENSE RATIO FUNDS (< 1%) ---")
print(pd.read_sql_query(q3, con=engine))

📊 Running Analytical Insights...

--- TOP 5 FUNDS BY AUM ---
Empty DataFrame
Columns: [amfi_code, total_aum]
Index: [] 

--- AVERAGE NAV PER MONTH (First 5 Rows) ---
     month     avg_nav
0  2022-01  207.061394
1  2022-02  207.717759
2  2022-03  209.692614
3  2022-04  211.833458
4  2022-05  212.731451 

--- LOW EXPENSE RATIO FUNDS (< 1%) ---
Empty DataFrame
Columns: [amfi_code, expense_ratio]
Index: []
